<a href="https://colab.research.google.com/github/cjsdudwls1/2026_Spring_Data_Standardization/blob/main/n%2B1%EC%A3%BC%EC%B0%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import warnings
from sklearn.datasets import fetch_20newsgroups
!pip install gensim
import gensim
from gensim import corpora
from gensim.models import LdaModel
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer

# pyLDAvis 시각화 라이브러리
!pip install pyLDAvis
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# 경고 메시지 숨기기
warnings.filterwarnings('ignore', category=DeprecationWarning)
nltk.download('stopwords', quiet=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 28.0 MB/s eta 0:00:00


True

In [7]:
# 1. 데이터 로드 및 무작위 5개 주제 추출

all_categories = ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc'
]

# 무작위로 5개 주제 선정
random.seed(42)
selected_categories = random.sample(all_categories, 3)

print(f"★ 정답 (무작위 선택된 5개 주제): {selected_categories}")

# 정답 데이터 로드 (시각화 분석 후 맞추기 위해 텍스트만 활용)
newsgroups = fetch_20newsgroups(subset='train', categories=selected_categories,
remove=('headers', 'footers', 'quotes'))
documents = newsgroups.data

★ 정답 (무작위 선택된 5개 주제): ['comp.sys.ibm.pc.hardware', 'alt.atheism', 'rec.motorcycles']


In [8]:
# 2. 텍스트 전처리
tokenizer = RegexpTokenizer(r'\w+')
en_stopwords = set(stopwords.words('english'))

# 추가적인 무의미한 단어 필터링
add_stopwords = {'ax', 'max', 'g9v', 'b8f', 'a86', 'pl', '145', '1d9', '0t', '34u'}
en_stopwords = en_stopwords.union(add_stopwords)

processed_docs = []
for doc in documents:
# 소문자 변환 및 토큰화
  tokens = tokenizer.tokenize(doc.lower())
  # 불용어 제거 및 3글자 이상 단어만 유지 (숫자 제외)
  stopped_tokens = [t for t in tokens if t not in en_stopwords and not t.isdigit() and len(t) > 2]
  processed_docs.append(stopped_tokens)

# Dictionary 및 Corpus 생성
dictionary = corpora.Dictionary(processed_docs)
# 지나치게 빈도가 낮거나 높은 단어 필터링 (품질 향상)
dictionary.filter_extremes(no_below=3, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in processed_docs]


In [10]:
# 3. 토픽 개수별(3개 ~ 7개) LDA 모델 학습 및 시각화 저장

for num_topics in range(3, 8):
    print(f"\n▶ 토픽 개수 [{num_topics}개] 모델 학습 및 시각화 생성 중...")
    # LDA 모델 학습
    lda_model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=100,
        update_every=1,
        chunksize=100,
        passes=10,
        alpha='auto',
        per_word_topics=True
    )
    # pyLDAvis 준비
    vis_data = gensimvis.prepare(lda_model, corpus, dictionary, sort_topics=False)

    # HTML 파일로 저장
    file_name = f'lda_vis_{num_topics}_topics.html'
    pyLDAvis.save_html(vis_data, file_name)


▶ 토픽 개수 [3개] 모델 학습 및 시각화 생성 중...

▶ 토픽 개수 [4개] 모델 학습 및 시각화 생성 중...

▶ 토픽 개수 [5개] 모델 학습 및 시각화 생성 중...

▶ 토픽 개수 [6개] 모델 학습 및 시각화 생성 중...

▶ 토픽 개수 [7개] 모델 학습 및 시각화 생성 중...
